# 02 - Preprocesamiento y Preparación del Pipeline de Datos
## Mushroom Toxicity Classification

En este cuaderno se implementa y valida el pipeline completo de preparación de datos:
1. **Carga de datos:** 61,069 instancias y 20 características (numéricas y categóricas).
2. **Manejo de valores faltantes (Missing Values):** Imputación en `cap-surface`, `gill-attachment`, `gill-spacing`, etc.
3. **Codificación y Escalado:** `OneHotEncoder` para nominales y `StandardScaler` para métricas (`cap-diameter`, `stem-height`, `stem-width`).
4. **Partición de datos:** División estratificada en **Entrenamiento (70%)**, **Validación (15%)** y **Prueba (15%)**.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configurar ruta a módulos del proyecto
sys.path.append(str(Path.cwd().parent))
from src.data_loader import (
    load_raw_data,
    get_train_val_test_data,
    NUMERIC_COLUMNS,
    TARGET_COLUMN
)
from src.models import build_preprocessing_pipeline

### 1. Carga del Dataset Completo

In [2]:
df = load_raw_data()
print(f"Filas totales (instancias): {df.shape[0]:,}")
print(f"Columnas totales (20 features + target): {df.shape[1]}")
df.head()

Filas totales (instancias): 61,069
Columnas totales (20 features + target): 21


,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,p,15.26,x,g,o,f,e,NaN,w,16.95,...,s,y,w,u,w,t,g,NaN,d,w
1,p,16.60,x,g,o,f,e,NaN,w,17.99,...,s,y,w,u,w,t,g,NaN,d,u
2,p,14.07,x,g,o,f,e,NaN,w,17.80,...,s,y,w,u,w,t,g,NaN,d,w
3,p,14.17,f,h,e,f,e,NaN,w,15.77,...,s,y,w,u,w,t,p,NaN,d,w
4,p,14.64,x,h,o,f,e,NaN,w,16.53,...,s,y,w,u,w,t,p,NaN,d,w


### 2. Detección de Valores Nulos (Missing Values)

In [3]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({
    'Valores Nulos': null_counts,
    'Porcentaje (%)': null_pct
}).sort_values('Porcentaje (%)', ascending=False)

missing_report[missing_report['Valores Nulos'] > 0]

,Valores Nulos,Porcentaje (%)
veil-type,57892,94.80
spore-print-color,54715,89.60
veil-color,53656,87.86
stem-root,51538,84.39
stem-surface,38124,62.43
gill-spacing,25063,41.04
cap-surface,14120,23.12
gill-attachment,9884,16.18
ring-type,2471,4.05


### 3. Partición Estratificada (Train / Validation / Test)

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = get_train_val_test_data(
    df, train_size=0.70, val_size=0.15, test_size=0.15, random_state=42
)

print(f"Conjunto de Entrenamiento: {X_train.shape[0]:,} filas ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Conjunto de Validación:    {X_val.shape[0]:,} filas ({X_val.shape[0]/len(df)*100:.1f}%)")
print(f"Conjunto de Prueba:        {X_test.shape[0]:,} filas ({X_test.shape[0]/len(df)*100:.1f}%)")

# Comprobar estratificación de clases
print("\nDistribución de clases (p / e):")
print("Train:\n", y_train.value_counts(normalize=True).round(4))
print("Val:\n", y_val.value_counts(normalize=True).round(4))
print("Test:\n", y_test.value_counts(normalize=True).round(4))

Conjunto de Entrenamiento: 42,748 filas (70.0%)
Conjunto de Validación:    9,160 filas (15.0%)
Conjunto de Prueba:        9,161 filas (15.0%)

Distribución de clases (p / e):
Train:
 class
p    0.5549
e    0.4451
Name: proportion, dtype: float64
Val:
 class
p    0.5549
e    0.4451
Name: proportion, dtype: float64
Test:
 class
p    0.555
e    0.445
Name: proportion, dtype: float64


### 4. Construcción y Aplicación del Pipeline de Preprocesamiento

In [5]:
categorical_cols = [col for col in X_train.columns if col not in NUMERIC_COLUMNS]

preprocessor = build_preprocessing_pipeline(NUMERIC_COLUMNS, categorical_cols)

# Ajustar únicamente en entrenamiento y transformar los tres conjuntos
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(f"Dimensiones de la matriz transformada (Train): {X_train_proc.shape}")
print(f"Dimensiones de la matriz transformada (Val):   {X_val_proc.shape}")
print(f"Dimensiones de la matriz transformada (Test):  {X_test_proc.shape}")

Dimensiones de la matriz transformada (Train): (42748, 128)
Dimensiones de la matriz transformada (Val):   (9160, 128)
Dimensiones de la matriz transformada (Test):  (9161, 128)
